# RQ1 — Reentrant clustering

In [ ]:
from pathlib import Path
import os
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
plt.rcParams.update(STYLE.mpl_rc())
OUTPUT_ROOT = Path(os.environ.get("RNA_ANALYSIS_RESULTS_ROOT", str(ROOT / "analysis_results"))).expanduser()

SAVE_FIGURES = True
FIGURE_ROOT = OUTPUT_ROOT / "50b_reentrant_full" / "figures"
if SAVE_FIGURES:
    FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename: str) -> None:
    """Save a notebook figure alongside its sweep's numerical results."""
    if SAVE_FIGURES:
        fig.savefig(FIGURE_ROOT / filename, dpi=250, bbox_inches="tight")


In [ ]:
from src.plots.figures import (
    plot_cluster_bins, plot_contact_balance, plot_metric_points,
    plot_recurrence_summary
)

SWEEP = "50b_reentrant_full"
DENSITY = 0.01
RECURRENCE_STIFFNESS = 2.5

summary = filter_rows(load_sweep(OUTPUT_ROOT, SWEEP, "summary"), density=DENSITY)
replicates = filter_rows(load_sweep(OUTPUT_ROOT, SWEEP, "replicates"), density=DENSITY)
E_VALUES = sorted(summary["E_att"].dropna().unique())
STIFFNESSES = sorted(summary["angle_k_ppp"].dropna().unique())
print(f"ρ = {DENSITY:g} | {len(STIFFNESSES)} stiffness values | {len(E_VALUES)} attraction strengths")

## Cluster-size distribution

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=partial(plot_cluster_bins, scheme="molecule_count", style=STYLE),
    style=STYLE, ncols=3, width=3.95, height=2.65,
    sharey=True, legend="bottom",
)
for col, ax in enumerate(axes.flat):
    ax.set_ylabel("Mean fraction" if col == 0 else "")
    ax.set_ylim(0, 1.0)
fig.legends[0].set_title("Cluster size bin")
save_figure(fig, "reentrance_01.png")
plt.show()

## Intra-, inter- and total base-pair contacts

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=partial(plot_contact_balance, style=STYLE, x_order=E_VALUES),
    style=STYLE, ncols=3, width=3.95, height=3.4,
    sharey=True, legend="bottom",
)
for col, ax in enumerate(axes.flat):
    if col > 0:
        ax.set_ylabel("")
save_figure(fig, "reentrance_02.png")
plt.show()

## Molecular contact graph

In [ ]:
METRICS = (
    ("mean_edge_weight", "Edge weight", "Mean edge weight"),
    ("n_edges", "Total edge count", "Mean edges per frame"),
)
for stiffness in STIFFNESSES:
    subset = filter_rows(replicates, angle_k_ppp=stiffness)
    fig, axes = plt.subplots(1, 2, figsize=(8.1, 3.2), layout="constrained")
    for ax, (metric, heading, ylabel) in zip(axes, METRICS):
        plot_metric_points(subset, ax, metric=metric, style=STYLE)
        low, high = subset[metric].min(), subset[metric].max()
        ax.set_ylim(
            max(0, low - 1.5) if metric == "mean_edge_weight" else 0,
            high + (2 if metric == "mean_edge_weight" else 70),
        )
        ax.set_title(heading)
        ax.set_ylabel(ylabel)
    fig.suptitle(rf"$k_{{\mathrm{{bbb}}}}={stiffness:g}$")
    save_figure(fig, "reentrance_03.png")
plt.show()

## Molecule-contact recurrence

In [ ]:
selected_summary = filter_rows(summary, angle_k_ppp=RECURRENCE_STIFFNESS)
selected_replicates = filter_rows(replicates, angle_k_ppp=RECURRENCE_STIFFNESS)
fig, ax = plt.subplots(figsize=(6.6, 4.2), layout="constrained")
plot_recurrence_summary(
    selected_summary, ax, replicates=selected_replicates, style=STYLE
)
ax.set_title(rf"$k_{{\mathrm{{bbb}}}}={RECURRENCE_STIFFNESS:g}$")
from matplotlib.lines import Line2D
legend = [
    Line2D([], [], marker="o", linestyle="none", color="#75889B",
           label="Replicates"),
    Line2D([], [], color="#235F92", lw=2, label="Mean ± SD"),
]
fig.set_layout_engine(None)
fig.subplots_adjust(bottom=.25)
fig.legend(handles=legend, loc="lower center",
           bbox_to_anchor=(.5, .04), ncol=2, frameon=True,
           fancybox=True, edgecolor="#D5D5D5", framealpha=.96)
save_figure(fig, "reentrance_04.png")
plt.show()

## Final-frame XY projections

The original GSD-based code below is optional (`SHOW_PROJECTIONS = False`). For the published notebook, embed the already-exported final-frame projection PNG here rather than requiring readers to download trajectories.


In [ ]:
SHOW_PROJECTIONS = False
if SHOW_PROJECTIONS:
    from src.plots.projections import plot_projection_sweep

    GSD_ROOT = ROOT.parent / "validation" / "bulk" / "output_dir"
    fig, axes = plot_projection_sweep(
        GSD_ROOT / SWEEP, row_by="angle_k_ppp",
        density=DENSITY, attractions=(2, 4, 8), style=STYLE,
    )
    figure_file = FIGURE_ROOT / "reentrance_xy.png"
    figure_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_file, dpi=250, bbox_inches="tight")
    print(f"Saved: {figure_file}")
    save_figure(fig, "reentrance_05.png")
plt.show()